In [ ]:
import os
import random
import glob
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import hashlib
from tqdm import tqdm
import math

print("GPU:", tf.config.list_physical_devices('GPU'))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
IMG_SIZE = 192            
IMG_CHANNELS = 3
BATCH_SIZE = 128
NUM_CLASSES = 29  

CLASS_NAMES = [chr(i) for i in range(ord('A'), ord('Z') + 1)] + ['space', 'del', 'nothing']
CLASS_NAMES = sorted(CLASS_NAMES)
print(f"Broj klasa: {len(CLASS_NAMES)}")
print(CLASS_NAMES)

DATA_ROOT = "/kaggle/input/datasets/debashishsau/aslamerican-sign-language-aplhabet-dataset/ASL_Alphabet_Dataset/asl_alphabet_train"
MODEL_OUT_PATH = "models/asl_cnn.keras"


In [ ]:
counts = {}
for cls in sorted(os.listdir(DATA_ROOT)):
    cls_dir = os.path.join(DATA_ROOT, cls)
    if os.path.isdir(cls_dir):
        counts[cls] = len(glob.glob(os.path.join(cls_dir, "*")))

counts_df = pd.DataFrame(sorted(counts.items()), columns=["class", "n_images"])
print(counts_df)

plt.figure(figsize=(12, 4))
plt.bar(counts_df["class"], counts_df["n_images"])
plt.xticks(rotation=90)
plt.title("Broj slika po klasi")
plt.tight_layout()
plt.show()

In [ ]:
def get_sample_images(data_root, class_names):
    samples = {}
    for class_name in class_names:
        class_dir = os.path.join(data_root, class_name)
        files = glob.glob(os.path.join(class_dir, "*"))
        if files:
            samples[class_name] = files[0]
    return samples

train_samples = get_sample_images(DATA_ROOT, CLASS_NAMES)

fig, axes = plt.subplots(5, 6, figsize=(15, 13))
axes = axes.flatten()

for i, class_name in enumerate(CLASS_NAMES):
    if class_name in train_samples:
        image = plt.imread(train_samples[class_name])
        axes[i].imshow(image)
    else:
        axes[i].text(0.5, 0.5, "nema slike", ha="center", va="center")

    axes[i].set_title(class_name)
    axes[i].axis("off")

for i in range(len(CLASS_NAMES), len(axes)):
    axes[i].axis("off")

plt.suptitle("Primjer slike iz svake klase", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
tqdm.pandas()

HASH_CSV_PATH = "/kaggle/input/datasets/vedrandragani/dfhash/df_with_hash.csv"

if os.path.exists(HASH_CSV_PATH):
    print(f"Učitavam postojeći DataFrame iz: {HASH_CSV_PATH}")
    df = pd.read_csv(HASH_CSV_PATH)

else:
    print("CSV datoteka ne postoji.")

    filepaths = []
    labels = []

    for cls in CLASS_NAMES:
        cls_dir = os.path.join(DATA_ROOT, cls)

        for fp in glob.glob(os.path.join(cls_dir, "*")):
            filepaths.append(fp)
            labels.append(cls)

    df = pd.DataFrame({
        "filepath": filepaths,
        "label": labels
    })

    print("Ukupno slika:", len(df))

    def file_hash(path):
        with open(path, "rb") as f:
            return hashlib.md5(f.read()).hexdigest()

    df["hash"] = df["filepath"].progress_apply(file_hash)

    os.makedirs("data", exist_ok=True)
    df.to_csv(HASH_CSV_PATH, index=False)

    print(f"DataFrame spremljen u: {HASH_CSV_PATH}")

print("Ukupno slika:", len(df))

In [ ]:
print("Prije dedupe:", len(df))

df_deduped = df.drop_duplicates(subset='hash', keep='first').reset_index(drop=True)

print("Nakon uklanjanja duplikata:", len(df_deduped))
print("\nBroj slika po klasi:")

df = df_deduped

counts_df = (
    df_deduped['label']
    .value_counts()
    .reset_index()
)
counts_df.columns = ["class", "n_images"]

counts_df = counts_df.sort_values("class").reset_index(drop=True)

print(counts_df)

plt.figure(figsize=(12, 4))
plt.bar(counts_df["class"], counts_df["n_images"])
plt.xticks(rotation=90)
plt.title("Broj slika po klasi (nakon uklanjanja duplikata)")
plt.tight_layout()
plt.show()

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label"], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED
)

print("Train:", len(train_df), " Val:", len(val_df), " Test:", len(test_df))

label_to_index = {name: i for i, name in enumerate(CLASS_NAMES)}
index_to_label = {i: name for name, i in label_to_index.items()}

In [ ]:
CROP_FRAC = 0.015

def trim_fixed_border(img):
    shape = tf.shape(img)
    h, w = shape[0], shape[1]

    crop_h = tf.cast(tf.cast(h, tf.float32) * CROP_FRAC, tf.int32)
    crop_w = tf.cast(tf.cast(w, tf.float32) * CROP_FRAC, tf.int32)

    img = img[crop_h : h - crop_h, crop_w : w - crop_w, :]
    return img


def load_and_preprocess(filepath, label):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=IMG_CHANNELS)
    img = trim_fixed_border(img)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    img = tf.image.random_saturation(img, lower=0.8, upper=1.2)

    label_onehot = tf.one_hot(label, NUM_CLASSES)
    return img, label_onehot


def make_dataset(df, shuffle=False):
    paths = df["filepath"].values
    labels_idx = df["label"].map(label_to_index).values.astype(np.int32)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels_idx))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED, reshuffle_each_iteration=True)

    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df, shuffle=False)
test_ds = make_dataset(test_df, shuffle=False)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

In [ ]:
for images, labels_batch in train_ds.take(1):
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for i, ax in enumerate(axes.flat):
        ax.imshow(images[i].numpy())
        cls_idx = np.argmax(labels_batch[i].numpy())
        ax.set_title(index_to_label[cls_idx])
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
class RandomCutout(layers.Layer):
    def __init__(self, cutout_frac=1/6, probability=0.3, **kwargs):
        super().__init__(**kwargs)
        self.cutout_frac = cutout_frac
        self.probability = probability

    def call(self, images, training=None):
        if not training:
            return images

        img_size = tf.shape(images)[1]
        cut_size = tf.cast(tf.cast(img_size, tf.float32) * self.cutout_frac, tf.int32)

        def apply_cutout_to_one(img):
            apply = tf.random.uniform([]) < self.probability
            return tf.cond(apply, lambda: self._cutout_single(img, img_size, cut_size), lambda: img)

        return tf.map_fn(apply_cutout_to_one, images)

    def _cutout_single(self, img, img_size, cut_size):
        x = tf.random.uniform([], 0, img_size - cut_size, dtype=tf.int32)
        y = tf.random.uniform([], 0, img_size - cut_size, dtype=tf.int32)

        pad_top, pad_bottom = y, img_size - y - cut_size
        pad_left, pad_right = x, img_size - x - cut_size

        square = tf.zeros((cut_size, cut_size, IMG_CHANNELS))
        square = tf.pad(square, [[pad_top, pad_bottom], [pad_left, pad_right], [0, 0]],
                         constant_values=1.0)
        return img * square

In [ ]:
def get_augmentation():
    return keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomZoom(0.15),
        layers.RandomRotation(0.06),
        layers.RandomTranslation(0.08, 0.08),
        layers.RandomBrightness(0.2, value_range=(0.0, 1.0)),
        layers.RandomContrast(0.2),
        RandomCutout(cutout_frac=1/6, probability=0.4),
    ], name="augmentation")

def build_transfer_model(input_shape=(192, 192, 3), num_classes=29):
    base_model = MobileNetV2(input_shape=input_shape, include_top=False, weights="imagenet")
    base_model.trainable = False

    augmentation = get_augmentation()

    inputs = keras.Input(shape=input_shape)
    x = augmentation(inputs)
    x = layers.Rescaling(255.0)(x)
    x = preprocess_input(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs)
    return model, base_model

model, base_model = build_transfer_model()

In [ ]:
sample_batch = next(iter(train_ds))
images, _ = sample_batch

aug_layer = model.get_layer("augmentation")
augmented = aug_layer(images[:8], training=True)

print("Min/Max nakon augmentacije:", augmented.numpy().min(), augmented.numpy().max())

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    ax.imshow(np.clip(augmented[i].numpy(), 0, 1))
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
LEARNING_RATE = 1e-3

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"],
    steps_per_execution=16
)

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    min_delta=0.001, 
    restore_best_weights=True,
    verbose=1,
)

model_checkpoint = keras.callbacks.ModelCheckpoint(
    MODEL_OUT_PATH,
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,            
    patience=2,             
    min_delta=0.001,
    cooldown=1,            
    min_lr=1e-6,
    verbose=1,
)

callbacks = [
    early_stopping,
    reduce_lr,
    model_checkpoint
]

y_train_labels = train_df['label'].map(label_to_index).values
weights = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_train_labels)
class_weight_dict = dict(enumerate(weights))

for cls in ['M', 'N', 'A', 'S', 'T', 'U', 'V', 'K', 'J', 'Z', 'G', 'H', 'del']:
    idx = label_to_index[cls]
    class_weight_dict[idx] *= 1.5

In [ ]:
model.summary()

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks,
    class_weight=class_weight_dict
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

history_data = history.history

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)

plt.plot(
    history_data["accuracy"],
    label="Train accuracy"
)

plt.plot(
    history_data["val_accuracy"],
    label="Validation accuracy"
)

plt.title("Accuracy tijekom treniranja")
plt.xlabel("Epoha")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)

plt.plot(
    history_data["loss"],
    label="Train loss"
)

plt.plot(
    history_data["val_loss"],
    label="Validation loss"
)

plt.title("Loss tijekom treniranja")
plt.xlabel("Epoha")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

EPOCHS_FT = 50
steps_per_epoch = math.ceil(len(train_df) / BATCH_SIZE)

lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-5,
    decay_steps=EPOCHS_FT * steps_per_epoch,
)

model.compile(
    optimizer=keras.optimizers.Adam(lr_schedule),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"],
)

callbacks_ft = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint(MODEL_OUT_PATH, monitor="val_loss", save_best_only=True, verbose=1),
]

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FT,
    callbacks=callbacks_ft,
    class_weight=class_weight_dict, 
)

In [ ]:
history_data = history_ft.history

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)

plt.plot(
    history_data["accuracy"],
    label="Train accuracy"
)

plt.plot(
    history_data["val_accuracy"],
    label="Validation accuracy"
)

plt.title("Accuracy tijekom treniranja")
plt.xlabel("Epoha")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)

plt.plot(
    history_data["loss"],
    label="Train loss"
)

plt.plot(
    history_data["val_loss"],
    label="Validation loss"
)

plt.title("Loss tijekom treniranja")
plt.xlabel("Epoha")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
test_loss, test_accuracy = model.evaluate(
    test_ds,
    verbose=1
)

print(f"\nTest loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

y_true = []
y_pred = []

for images, labels in tqdm(
    test_ds,
    desc="Predviđanje test skupa",
    unit="batch"
):
    predictions = model.predict(images, verbose=0)

    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(predictions, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 score:  {f1:.4f}")

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred
)

plt.figure(figsize=(14, 14))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CLASS_NAMES
)

disp.plot(
    ax=plt.gca(),
    xticks_rotation=90,
    cmap="Blues",
    colorbar=True
)

plt.title("Confusion matrix")
plt.tight_layout()
plt.show()

In [ ]:
y_true = []
y_pred = []

for images, labels_batch in test_ds:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(np.argmax(labels_batch.numpy(), axis=1))

print(classification_report(
    y_true, y_pred, target_names=CLASS_NAMES, zero_division=0
))

In [ ]:
model.save_weights("models/asl_cnn.weights.h5")